# Multiagent Pipeline

In [7]:
# Facoltativo: installa dipendenze se ti servono in un ambiente pulito
# %pip install -r requirements.txt

# Configuration

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

LM_STUDIO_BASE_URL = "https://api.mistral.ai/v1"
LM_STUDIO_API_KEY  = os.getenv("MISTRAL_API_KEY", "")
LM_STUDIO_MODEL    = "mistral-small-latest"

# Paths 
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
ALLARMI_CSV = os.path.join(DATA_DIR, "ALLARMI.csv")
TIPOLOGIA_CSV = os.path.join(DATA_DIR, "TIPOLOGIA_VIAGGIATORE.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
FINDINGS_JSON = os.path.join(OUTPUT_DIR, "findings.json")

# Outlier Detection 
DEFAULT_OUTLIER_ALGORITHM = "IsolationForest"
ISOLATION_FOREST_CONTAMINATION = 0.05
LOF_N_NEIGHBORS = 20
ZSCORE_THRESHOLD = 3.0

# Risk Thresholds 
ALERT_RATE_MULTIPLIER = 3.0


# Tool REPL

In [9]:
import sys
import io
import traceback
from typing import Annotated
from langchain_core.tools import tool


class PythonREPL:
    """Executes Python code and captures output."""
    def __init__(self):
        self.globals = {}

    def run(self, code: str) -> str:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout = mystdout = io.StringIO()
        sys.stderr = mystderr = io.StringIO()
        try:
            exec(code, self.globals)
            output = mystdout.getvalue()
            errors = mystderr.getvalue()
            if errors:
                output += f"\nSTDERR:\n{errors}"
            return output if output else "(OK, no output)"
        except Exception:
            return f"Error:\n{traceback.format_exc()}"
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr


_repl = PythonREPL()


# Prompts
Natural language prompts for each pipeline task.
The LLM receives these and generates Python code autonomously.

In [ ]:
# ── Findings helper ──────────────────────────────────────────────────────────

def _findings_guidance(task_key: str, extra_notes: str = "") -> str:
    base = (
        f"Maintain a shared findings JSON at '{FINDINGS_JSON}'. "
        f"At the start, attempt to load it; if missing, empty, invalid, or corrupted, initialize an empty dict instead of failing. "
        f"Store new information under the key '{task_key}' while preserving existing keys for other tasks. "
        f"Use concise, machine-readable fields with only native Python JSON-serializable types such as dict, list, str, int, float, bool, or null. "
        f"Convert pandas and numpy values to native Python types before saving. "
        f"Convert tuples to lists before saving. "
        f"After completing the task, update the entry for '{task_key}' and write the full JSON back by overwriting the file. "
    )
    if extra_notes:
        base += extra_notes
    return base


# ── Task 1: Clean the dataset ─────────────────────────────────────────────────

def _build_structural_cleaning_prompt(input_path, output_path):
    return (
        f"You are a data cleaning agent for tabular datasets used in downstream merge and anomaly detection tasks. "
        f"Load the dataset at '{input_path}' and produce a cleaned version saved to '{output_path}'. "
        f"Work autonomously and infer the required Python libraries. "
        f"First inspect the raw schema and print the original shape and original column names. "
        f"Then standardize column names to lowercase snake_case. "
        f"Apply column-name normalization at the token or word level, not at the individual-character level. "
        f"Do not insert separators between consecutive characters when a name is already a single continuous token written in uppercase or lowercase. "
        f"Convert naming styles consistently by identifying word boundaries, replacing non-alphanumeric separators with underscores, collapsing repeated underscores, and trimming leading or trailing underscores. "
        f"Immediately after renaming, resolve duplicate column names deterministically by occurrence order. "
        f"When duplicate column labels are present, rebuild the full ordered list of column names so that each occurrence gets a unique stable name. "
        f"Do not use ambiguous renaming by duplicated label alone. "
        f"Do not continue until column names are truly unique and explicitly verified. "
        f"After the schema is safe, clean values column by column. "
        f"The cleaned dataset must have all column names in lowercase snake_case and all textual values in lowercase. "
        f"This is a mandatory output requirement for the cleaned file. "
        f"For every column, apply text normalization to all values that are textual, regardless of how the column type is internally represented. "
        f"Do not rely on a specific column type such as 'object' to detect text. "
        f"Instead, treat any value that behaves like text as textual content and normalize it. "
        f"Convert all textual values to lowercase, remove outer whitespace, and normalize repeated internal whitespace. "
        f"Preserve null values as null. "
        f"If a column contains both text and non-text values, normalize the textual cells to lowercase and leave the non-text cells unchanged. "
        f"Do not skip lowercase normalization for textual values because of mixed content in the same column. "
        f"Do not change the semantic type of non-text values only for convenience. "
        f"The lowercase normalization of textual values must be applied consistently across all datasets, even if different internal data types are used. "
        f"Use one shared and globally defined set of missing-value patterns before any column-level cleaning logic starts. "
        f"Do not define cleaning rules or missing-value rules only inside conditional branches if they are reused elsewhere. "
        f"Keep the cleaning flow structurally simple and deterministic across datasets. "
        f"Use the same cleaning strategy for every dataset of this task category, rather than inventing dataset-specific logic unless strictly necessary for loadability. "
        f"Avoid branch-specific variables that may be unavailable in other branches. "
        f"The generated code must run top-to-bottom without relying on variables defined only inside a conditional block. "
        f"Detect values that semantically represent missing, undefined, or non-informative entries by analyzing the column content, and convert them into proper missing values where appropriate. "
        f"Then remove duplicate rows. "
        f"Handle missing values conservatively and preserve missingness when imputation could distort downstream anomaly detection. "
        f"Before saving, validate that the dataframe is non-empty and has unique column names. "
        f"Do not save any output if those checks fail. "
        f"Print only a short summary with: original shape, final shape, final column names, duplicate-name collisions handled, and duplicate rows removed. "
    )


def _build_data_prompt_1():
    return (
        _build_structural_cleaning_prompt(
            ALLARMI_CSV,
            f"{OUTPUT_DIR}/allarmi_clean.csv"
        )
        + _findings_guidance(
            "data_loading_allarmi",
            "Capture shape_before, shape_after, columns_final, duplicate_name_collisions, duplicate_rows_removed. "
        )
    )


def _build_data_prompt_2():
    return (
        _build_structural_cleaning_prompt(
            TIPOLOGIA_CSV,
            f"{OUTPUT_DIR}/tipologia_clean.csv"
        )
        + _findings_guidance(
            "data_loading_tipologia",
            "Capture shape_before, shape_after, columns_final, duplicate_name_collisions, duplicate_rows_removed. "
        )
    )


# ── Task 2: Normalize the dataset ─────────────────────────────────────────────

def _build_semantic_normalization_prompt(input_path, output_path, findings_key):
    return (
        f"You are a semantic normalization agent for tabular datasets already cleaned at schema level. "
        f"Load the dataset at '{input_path}' and produce a semantically refined version saved to '{output_path}'. "
        f"Work autonomously and infer the required Python libraries. "
        f"The dataset already has a safe schema, so do not rename, reorder, merge, split, or drop columns unless strictly necessary. "
        f"Focus only on intra-column semantic consistency. "
        f"For each column, inspect non-null values and infer the dominant representation. "
        f"Ignore missing or non-informative values when inferring patterns. "
        f"Normalize only high-confidence minority variants. "
        f"Do not collapse distinct categories. "
        f"Preserve uncertain or difform values and report them. "
        f"Before saving, validate that the dataframe is non-empty and still loadable. "
        f"The dataset must be loaded, processed, and saved within the same execution flow. "
        f"All steps must run automatically without requiring any explicit invocation or entry point. "
        f"If validation passes, write the output file immediately using to_csv. "
        f"Do not check whether the output file already exists before writing it. "
        f"The output file must always be created during execution if the dataframe is valid. "
        f"Save the semantically refined dataset to '{output_path}' without index. "
        + _findings_guidance(
            findings_key,
            "Include shape_before, shape_after, normalization_mappings, preserved_difform_values. "
        )
    )


# ── Task 3: Merge ─────────────────────────────────────────────────────────────

def _build_merge_prompt():
    return (
        f"You are a data integration agent responsible for combining two cleaned tabular datasets into a single unified dataset for downstream anomaly detection. "
        f"Load the dataset at '{OUTPUT_DIR}/allarmi_clean.csv' and the dataset at '{OUTPUT_DIR}/tipologia_clean.csv' using pandas. "
        f"Work autonomously and infer the required Python libraries. "
        f"First, inspect both dataframes independently: print their shapes and column names. "
        f"Then identify the common columns between the two dataframes and print them explicitly before proceeding. "
        f"Merge the two dataframes on the common columns using an outer join to preserve all records from both sources. "
        f"After merging, ensure that no duplicate columns remain in the result. "
        f"Do not drop or rename any column unless it is a confirmed structural duplicate introduced by the merge. "
        f"Do not perform any imputation, encoding, or transformation on the merged data. "
        f"Preserve the original values and types from both sources as-is. "
        f"Before saving, validate that the merged dataframe is non-empty and has unique column names. "
        f"Do not save any output if those checks fail. "
        f"Save the merged dataset to '{OUTPUT_DIR}/merged_data.csv' without index. "
        f"Ensure that the dataset is loaded, processed, and saved within the same execution flow. "
        f"Avoid defining execution entry points or structures that require explicit invocation. "
        f"Assume that the code will be executed exactly as written, so all steps must run immediately. "
        f"Print only a short summary with: shape of allarmi_clean, shape of tipologia_clean, common columns used for merge, final merged shape, and duplicate columns removed. "
        + _findings_guidance(
            "merge",
            "Record shapes_allarmi_tipologia, common_columns, merged_shape, duplicate_columns_removed. "
        )
    )


# ── Task 4: Group by route ────────────────────────────────────────────────────

def _build_baseline_prompt():
    return (
        f"You are a data aggregation agent responsible for building a route-level summary dataset to be used as baseline input for downstream anomaly detection. "
        f"Load the dataset at '{OUTPUT_DIR}/merged_data.csv'. "
        f"Work autonomously and infer the required Python libraries. "
        f"First, inspect the loaded dataframe: print its shape and column names before proceeding. "
        f"Create a 'route' column by combining the values of 'areoporto_partenza' and 'areoporto_arrivo'. "
        f"Do not drop the original airport columns after creating the route column. "
        f"Build the aggregation strategy for every column except 'route'. "
        f"Do not hardcode column names when defining the aggregation strategy. "
        f"Determine whether a column is numeric using pandas-native type inspection, not numpy subtype checks. "
        f"For numeric columns, aggregate by summing their values. "
        f"For non-numeric columns, aggregate by taking the first observed value. "
        f"Do not apply any transformation or normalization to the aggregated values. "
        f"Group the dataframe by 'route' and aggregate all other columns according to the strategy above. "
        f"After aggregation, ensure that 'route' is present as a standard column in the final dataframe and not only as an index. "
        f"Before saving, validate that the resulting dataframe is non-empty and has unique column names. "
        f"Do not save any output if those checks fail. "
        f"Save the aggregated dataset to '{OUTPUT_DIR}/routes_summary.csv' without index. "
        f"Ensure that the dataset is loaded, processed, and saved within the same execution flow. "
        f"Avoid defining execution entry points or structures that require explicit invocation. "
        f"Assume that the code will be executed exactly as written, so all steps must run immediately. "
        f"Print only a short summary with: original merged shape, number of unique routes found, final routes_summary shape. "
        + _findings_guidance(
            "baseline_grouping",
            "Store merged_shape, unique_routes, aggregation_strategy_summary (list of numeric/non_numeric columns), routes_summary_shape. "
        )
    )


# ── Task 5: Baseline statistics ───────────────────────────────────────────────

def _build_baseline_stats_prompt():
    return (
        f"Load '{OUTPUT_DIR}/routes_summary.csv' with pandas. "
        f"Import numpy as np. "
        f"Ensure 'allarmati' is numeric using pd.to_numeric(errors='coerce').fillna(0). "
        f"Compute global mean and std of 'allarmati' across all routes. "
        f"Add column 'rolling_mean_alarms' = global mean. "
        f"Add column 'rolling_std_alarms' = global std. If std is 0, set it to 1. "
        f"Add column 'z_score': (allarmati - rolling_mean_alarms) / rolling_std_alarms. "
        f"Add column 'ratio_to_baseline': allarmati / rolling_mean_alarms. Replace inf and -inf with 0. "
        f"Print global mean and std. "
        f"Print shape. "
        f"Print top 10 rows by z_score descending showing route, allarmati, rolling_mean_alarms, z_score. "
        f"Save the full dataframe to '{OUTPUT_DIR}/baseline_data.csv' without index. "
        + _findings_guidance(
            "baseline_stats",
            "Store global_mean_allarmati, global_std_allarmati, baseline_shape, top_routes_by_z (list of dicts with route, allarmati, z_score). "
        )
    )


# ── Task 6: Outlier Detection ─────────────────────────────────────────────────

def _build_outlier_prompt():
    algo = DEFAULT_OUTLIER_ALGORITHM
    contam = ISOLATION_FOREST_CONTAMINATION
    neighbors = LOF_N_NEIGHBORS
    zscore_t = ZSCORE_THRESHOLD

    return (
        f"You are an outlier detection agent operating on a route-level baseline dataset. "
        f"Load the dataset at '{OUTPUT_DIR}/baseline_data.csv'. "
        f"If a shared findings JSON exists at '{FINDINGS_JSON}', load it and reuse relevant information from previous steps "
        f"(especially baseline statistics and column validation). Continue even if it is missing. "
        f"Work autonomously and infer the required Python libraries. "
        f"First inspect the dataset: print shape and column names. "
        f"Ensure that the columns representing engineered features are valid, numeric, and usable for modeling. "
        f"The expected features are 'allarmati', 'z_score', and 'ratio_to_baseline'. "
        f"Coerce invalid values to numeric form and handle non-finite values safely so that the dataset is model-ready. "
        f"Construct a feature matrix using exactly these three features, preserving row alignment with the original dataset. "
        + (
            f"Apply an Isolation Forest model using a contamination level of {contam} to detect anomalies. "
            if algo == "IsolationForest" else
            f"Apply a Local Outlier Factor model using {neighbors} neighbors and contamination {contam}. "
            if algo == "LOF" else
            f"Detect anomalies using a z-score threshold of {zscore_t} on the absolute value of z_score. "
        )
        + f"Store the result in a new boolean column named 'anomaly'. "
        f"Do not drop any columns or rows. Preserve the full dataset. "
        f"Print a short summary with: total number of rows and number of detected anomalies. "
        f"Print the top 10 anomalous rows sorted by highest anomaly severity using z_score, "
        f"displaying only 'route', 'allarmati', and 'z_score'. "
        f"Before saving, validate that the dataframe is non-empty and consistent. "
        f"If validation passes, save the dataset to '{OUTPUT_DIR}/outlier_results.csv' without index. "
        f"The dataset must be loaded, processed, and saved within the same execution flow. "
        f"All steps must run automatically without requiring explicit invocation. "
        + _findings_guidance(
            "outlier_detection",
            "Store total_rows as int, anomaly_rows as int, algorithm_used as string, "
            "feature_columns as list of strings, and top_anomalies as a list of plain Python dicts with route, allarmati, z_score. "
        )
    )


# ── Task 7: Risk Profiling ────────────────────────────────────────────────────

def _build_risk_prompt():
    mult = ALERT_RATE_MULTIPLIER
    return (
        f"Load '{OUTPUT_DIR}/outlier_results.csv' with pandas. "
        f"Import numpy as np. "
        f"Filter only rows where anomaly is True. Print how many. "
        f"If zero, save empty dataframe to '{OUTPUT_DIR}/risk_profiled.csv' and print 'No anomalies'. "
        f"Otherwise: "
        f"rule_route = anom['ratio_to_baseline'] > {mult}. "
        f"rule_zscore_high = anom['z_score'].abs() > 8. "
        f"rule_zscore_med = (anom['z_score'].abs() > 5) & (~rule_zscore_high). "
        f"conditions = [rule_route & rule_zscore_high, rule_route | rule_zscore_high, rule_zscore_med]. "
        f"choices = ['CRITICAL', 'HIGH', 'MEDIUM']. "
        f"anom['risk_level'] = np.select(conditions, choices, default='LOW'). "
        f"Print risk_level value_counts. "
        f"Save full dataframe to '{OUTPUT_DIR}/risk_profiled.csv' without index."
        + _findings_guidance(
            "risk_profiling",
            "Store anomaly_rows, risk_level_counts, rules_used (text). "
        )
    )


# ── Task 8: Report ────────────────────────────────────────────────────────────
# FIX: removed stray `s` after opening parenthesis

def _build_report_prompt():
    return (
        f"Load '{OUTPUT_DIR}/risk_profiled.csv' with pandas. "
        f"Sort by z_score descending and take top 5 rows. "
        f"Import OpenAI from openai. "
        f"Create client: client = OpenAI(base_url='{LM_STUDIO_BASE_URL}', api_key='{LM_STUDIO_API_KEY}'). "
        f"For each row, call client.chat.completions.create("
        f"model='{LM_STUDIO_MODEL}', "
        f"messages=[{{'role':'user','content':'Explain this transit anomaly in 2 sentences: ' + str(row.to_dict())}}], "
        f"max_tokens=150). "
        f"Get response text from response.choices[0].message.content. "
        f"Build a text report with header 'TRANSIT ANOMALY REPORT' and today's date. "
        f"Save report to '{OUTPUT_DIR}/anomaly_report.txt'. "
        f"Also save a JSON version with json.dump to '{OUTPUT_DIR}/anomaly_report.json'. "
        f"Print the report."
        + _findings_guidance(
            "report_generation",
            "Store top_anomalies_in_report (list), report_txt_path, report_json_path. "
        )
    )


# ── _build_tasks(): single source of truth ────────────────────────────────────
# FIX: replaces the three duplicate TASKS list definitions

def _build_tasks():
    return [
        ("data_loading_allarmi",   _build_data_prompt_1()),
        ("data_loading_tipologia", _build_data_prompt_2()),
        ("semantic_allarmi", _build_semantic_normalization_prompt(
            f"{OUTPUT_DIR}/allarmi_clean.csv",
            f"{OUTPUT_DIR}/allarmi_semantic.csv",
            "semantic_allarmi",
        )),
        ("semantic_tipologia", _build_semantic_normalization_prompt(
            f"{OUTPUT_DIR}/tipologia_clean.csv",
            f"{OUTPUT_DIR}/tipologia_semantic.csv",
            "semantic_tipologia",
        )),
        ("merge",             _build_merge_prompt()),
        ("baseline_grouping", _build_baseline_prompt()),
        ("baseline_stats",    _build_baseline_stats_prompt()),
        ("outlier_detection", _build_outlier_prompt()),
        ("risk_profiling",    _build_risk_prompt()),
        ("report_generation", _build_report_prompt()),
    ]


TASKS = _build_tasks()


# Multi-Agent System

3 agents: Supervisor (routing logic), Code_executor (direct LLM → REPL),
Validator (file check — no LLM needed).

Key design: NO tool calling. The LLM generates pure Python code,
and the code_executor runs it directly in a REPL.
- Includes rate limiting for free-tier APIs.
- Includes task caching: if a task's output file already exists, skip it.

In [11]:
import os
import time
from typing import Literal
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.types import Command
from openai import OpenAI


# Ensure output dir exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Direct LLM client (no LangChain, no tool calling)
_client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)
# NOTE: _repl is defined in the Tool REPL cell above — not re-instantiated here.

# Rate limiting
_SECONDS_BETWEEN_CALLS = 8  # 10 RPM = 1 every 6s, we use 8s for safety
_last_call_time = 0.0

# Verbose logging
VERBOSE = True  # Set to False once pipeline is stable

# Cache control
USE_CACHE = True


#  State
class AgentState(MessagesState):
    next: str
    current_task_index: int
    task_status: str
    retry_count: int


MAX_RETRIES = 4

# Maps each task name to the file it MUST produce.
EXPECTED_FILES = {
    "data_loading_allarmi":   os.path.join(OUTPUT_DIR, "allarmi_clean.csv"),
    "data_loading_tipologia": os.path.join(OUTPUT_DIR, "tipologia_clean.csv"),
    "semantic_allarmi":       os.path.join(OUTPUT_DIR, "allarmi_semantic.csv"),
    "semantic_tipologia":     os.path.join(OUTPUT_DIR, "tipologia_semantic.csv"),
    "merge":                  os.path.join(OUTPUT_DIR, "merged_data.csv"),
    "baseline_grouping":      os.path.join(OUTPUT_DIR, "routes_summary.csv"),
    "baseline_stats":         os.path.join(OUTPUT_DIR, "baseline_data.csv"),
    "outlier_detection":      os.path.join(OUTPUT_DIR, "outlier_results.csv"),
    "risk_profiling":         os.path.join(OUTPUT_DIR, "risk_profiled.csv"),
    "report_generation":      os.path.join(OUTPUT_DIR, "anomaly_report.txt"),
}


# Helper: check if task output already exists (cache)
def _task_is_cached(task_name: str) -> bool:
    if not USE_CACHE:
        return False
    filepath = EXPECTED_FILES.get(task_name, "")
    if not filepath:
        return False
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        size = os.path.getsize(filepath)
        print(f"  [cache] '{task_name}' → {os.path.basename(filepath)} already exists ({size:,} bytes), skipping.")
        return True
    return False


# Helper: rate-limited LLM call
def _call_llm(messages: list[dict], max_tokens: int = 1024) -> str:
    global _last_call_time

    elapsed = time.time() - _last_call_time
    if elapsed < _SECONDS_BETWEEN_CALLS:
        wait = _SECONDS_BETWEEN_CALLS - elapsed
        print(f"  [rate_limit] Waiting {wait:.0f}s...")
        time.sleep(wait)

    for attempt in range(3):
        try:
            _last_call_time = time.time()
            response = _client.chat.completions.create(
                model=LM_STUDIO_MODEL,
                messages=messages,
                max_tokens=max_tokens,
                temperature=0.0,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                backoff = 15 * (2 ** attempt)  # 15s, 30s, 60s
                print(f"  [rate_limit] 429 received, waiting {backoff}s (attempt {attempt+1}/3)...")
                time.sleep(backoff)
            else:
                return f"LLM_ERROR: {e}"

    return "LLM_ERROR: 429 — rate limit exceeded after 3 retries"


# Helper: clean code from LLM response
def _clean_code(raw: str) -> str:
    code = raw.strip()
    if code.startswith("```"):
        lines = code.split("\n")
        lines = lines[1:]  # drop opening fence
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        code = "\n".join(lines)
    return code.strip()


# Helper: ask LLM for code and run it
def _ask_and_run(task_prompt: str, retry_context: str = "") -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a Python data analyst. "
                "Reply with ONLY Python code, nothing else. "
                "No markdown, no backticks, no explanations. Just code."
            ),
        },
        {"role": "user", "content": task_prompt},
    ]

    if retry_context:
        messages.append({
            "role": "user",
            "content": f"Error: {retry_context}\nFix the code. Reply with ONLY the corrected Python code.",
        })

    raw = _call_llm(messages)

    if raw.startswith("LLM_ERROR"):
        return raw

    code = _clean_code(raw)

    if not code:
        return "LLM_ERROR: empty code response"

    if VERBOSE:
        print(f"\n  {'─'*40}")
        print(f"  GENERATED CODE:")
        print(f"  {'─'*40}")
        for i, line in enumerate(code.split("\n"), 1):
            print(f"  {i:3d} | {line}")
        print(f"  {'─'*40}\n")

    return _repl.run(code)


# Supervisor
def supervisor_node(state: AgentState) -> Command[Literal["code_executor", "validator", "__end__"]]:
    idx = state.get("current_task_index", 0)
    status = state.get("task_status", "pending")
    retries = state.get("retry_count", 0)

    if idx >= len(TASKS):
        return Command(goto="__end__", update={"next": "__end__"})

    task_name, task_prompt = TASKS[idx]

    if status == "pending" and _task_is_cached(task_name):
        next_idx = idx + 1
        if next_idx >= len(TASKS):
            return Command(goto="__end__", update={
                "current_task_index": next_idx, "task_status": "done",
            })
        return Command(
            goto="supervisor",
            update={
                "messages": [HumanMessage(content=f"Cached: {task_name}", name="supervisor")],
                "current_task_index": next_idx,
                "task_status": "pending",
                "retry_count": 0,
            },
        )

    if status in ("pending", "failed"):
        if status == "failed" and retries >= MAX_RETRIES:
            print(f"\n  SKIP '{task_name}' after {retries} retries\n")
            next_idx = idx + 1
            return Command(
                goto="__end__" if next_idx >= len(TASKS) else "supervisor",
                update={
                    "messages": [HumanMessage(content=f"Skipped {task_name}.", name="supervisor")],
                    "current_task_index": next_idx,
                    "task_status": "pending",
                    "retry_count": 0,
                },
            )
        return Command(
            goto="code_executor",
            update={
                "messages": [HumanMessage(content=task_prompt, name="supervisor")],
                "task_status": "executing",
            },
        )

    if status == "executing":
        expected = EXPECTED_FILES.get(task_name, "")
        return Command(
            goto="validator",
            update={
                "messages": [HumanMessage(content=expected, name="supervisor")],
                "task_status": "validating",
            },
        )

    if status == "validating":
        last = state["messages"][-1].content if state["messages"] else ""
        if last.startswith("APPROVED"):
            next_idx = idx + 1
            print(f"  ✓ Task '{task_name}' OK")
            if next_idx >= len(TASKS):
                return Command(goto="__end__", update={
                    "current_task_index": next_idx, "task_status": "done",
                })
            return Command(
                goto="supervisor",
                update={
                    "current_task_index": next_idx,
                    "task_status": "pending",
                    "retry_count": 0,
                },
            )
        else:
            return Command(
                goto="supervisor",
                update={"task_status": "failed", "retry_count": retries + 1},
            )

    return Command(goto="__end__", update={})


# Code Executor
def code_executor_node(state: AgentState) -> Command[Literal["supervisor"]]:
    idx = state.get("current_task_index", 0)
    retries = state.get("retry_count", 0)

    if idx >= len(TASKS):
        return Command(
            update={"messages": [HumanMessage(content="All tasks done.", name="code_executor")]},
            goto="supervisor",
        )

    task_name, task_prompt = TASKS[idx]

    retry_context = ""
    if retries > 0:
        last_msg = state["messages"][-1].content if state["messages"] else ""
        if "Error" in last_msg or "REJECTED" in last_msg:
            retry_context = last_msg

    print(f"  [code_executor] Running '{task_name}' (attempt {retries + 1})...")
    result = _ask_and_run(task_prompt, retry_context)

    if len(result) > 2000:
        result = result[:2000] + "\n... [truncated]"

    print(f"  [code_executor] Output: {result[:300]}")

    return Command(
        update={"messages": [HumanMessage(content=result, name="code_executor")]},
        goto="supervisor",
    )


# Validator (pure Python, no LLM needed)
def validator_node(state: AgentState) -> Command[Literal["supervisor"]]:
    filepath = state["messages"][-1].content if state["messages"] else ""
    filepath = filepath.strip()

    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        size = os.path.getsize(filepath)
        msg = f"APPROVED: {filepath} ({size:,} bytes)"
        print(f"  [validator] {msg}")
    else:
        if VERBOSE and os.path.isdir(OUTPUT_DIR):
            files = os.listdir(OUTPUT_DIR)
            print(f"  [validator] Files in output dir: {files}")
        msg = f"REJECTED: {filepath} not found or empty"
        print(f"  [validator] {msg}")

    return Command(
        update={"messages": [HumanMessage(content=msg, name="validator")]},
        goto="supervisor",
    )


# Graph
def build_graph():
    g = StateGraph(AgentState)
    g.add_node("supervisor", supervisor_node)
    g.add_node("code_executor", code_executor_node)
    g.add_node("validator", validator_node)
    g.add_edge(START, "supervisor")
    return g.compile()


In [12]:
ALGORITHM = None            # "IsolationForest", "LOF", "zscore", or None
VERBOSE_RUN = False
NO_CACHE = False
USER_QUERY = None           # e.g. "areoporto_partenza == 'FCO'" or natural language

from datetime import datetime
import os
import shutil
import pandas as pd
from langchain_core.messages import HumanMessage

# Update algorithm and cache flag
if ALGORITHM:
    DEFAULT_OUTLIER_ALGORITHM = ALGORITHM

USE_CACHE = not NO_CACHE
if NO_CACHE:
    print("  [cache] Cache disabled — all tasks will re-run.\n")

# Optional query filtering
if USER_QUERY:
    sample_df = pd.read_csv(ALLARMI_CSV, nrows=5)
    sample_df.columns = (
        sample_df.columns.str.lower().str.strip()
        .str.replace(' ', '_').str.replace('%', '_')
    )
    cols = sample_df.columns.unique()
    col_info = {
        c: sample_df[c].head(3).tolist()
        for c in cols
        if getattr(sample_df[c], "ndim", 1) == 1
    }

    _query_client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)
    system_msg = (
        "You translate user queries into pandas DataFrame .query() expressions. "
        "Reply with ONLY the query string, nothing else. No quotes around it. "
        "All column names are lowercase with underscores. "
        "Airport IATA codes: Fiumicino=FCO, Ciampino=CIA, Malpensa=MXP, Linate=LIN, "
        "Bergamo=BGY, Venezia=VCE, Bologna=BLQ, Napoli=NAP, Catania=CTA, Pisa=PSA, "
        "Bari=BRI, Palermo=PMO, Torino=TRN, Verona=VRN. "
        "Available columns and sample values: " + str(col_info)
    )
    resp = _query_client.chat.completions.create(
        model=LM_STUDIO_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": USER_QUERY},
        ],
        temperature=0.0, max_tokens=100,
    )
    pandas_query = resp.choices[0].message.content.strip()
    print(f"  User query: {USER_QUERY}")
    print(f"  LLM interpreted as: {pandas_query}")

    for csv_path in [ALLARMI_CSV, TIPOLOGIA_CSV]:
        dst = os.path.join(OUTPUT_DIR, os.path.basename(csv_path))
        shutil.copy2(csv_path, dst)
        df = pd.read_csv(dst)
        df.columns = (
            df.columns.str.lower().str.strip()
            .str.replace(' ', '_').str.replace('%', '_')
        )
        df = df.loc[:, ~df.columns.duplicated()]
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].str.strip()
        try:
            filtered = df.query(pandas_query)
            filtered.to_csv(dst, index=False)
            print(f"  Filtered {os.path.basename(csv_path)}: {len(filtered)} rows")
        except Exception as e:
            print(f"  Filter failed: {e} — keeping original data")
            df.to_csv(dst, index=False)

    # Update paths so prompt builders pick up the filtered files
    ALLARMI_CSV = os.path.join(OUTPUT_DIR, "ALLARMI.csv")
    TIPOLOGIA_CSV = os.path.join(OUTPUT_DIR, "TIPOLOGIA_VIAGGIATORE.csv")

# Rebuild TASKS once (picks up any path/algorithm changes above)
TASKS = _build_tasks()

# Run
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("  TRANSIT ANOMALY DETECTION — MULTI-AGENT (Notebook)")
print("=" * 60)
print(f"  Time:       {datetime.now().strftime('%H:%M:%S')}")
print(f"  Model:      {LM_STUDIO_MODEL}")
print(f"  Algorithm:  {DEFAULT_OUTLIER_ALGORITHM}")
print(f"  Output:     {OUTPUT_DIR}")
print("=" * 60 + "\n")

graph = build_graph()

state = {
    "messages": [HumanMessage(content="Start pipeline.")],
    "current_task_index": 0,
    "task_status": "pending",
    "next": "supervisor",
    "retry_count": 0,
}

for event in graph.stream(state, {"recursion_limit": 80}):
    for node, data in event.items():
        if node == "__end__":
            continue
        msgs = data.get("messages", [])
        if msgs:
            content = msgs[-1].content if hasattr(msgs[-1], "content") else str(msgs[-1])
            if VERBOSE_RUN:
                print(f"\n{'='*50}")
                print(f"  [{node.upper()}]")
                print(f"{'='*50}")
                if len(content) > 1000:
                    content = content[:1000] + "\n... [truncated]"
                print(content)
            else:
                line = content.split("\n")[0][:100]
                print(f"  [{node}] {line}")

print("\n" + "=" * 60)
print("  DONE")
print("=" * 60)
for f in [
    "allarmi_clean.csv", "tipologia_clean.csv", "merged_data.csv",
    "routes_summary.csv", "baseline_data.csv", "outlier_results.csv",
    "risk_profiled.csv", "anomaly_report.txt", "anomaly_report.json",
]:
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(p):
        print(f"  ✓ {f} ({os.path.getsize(p):,} bytes)")
    else:
        print(f"  ✗ {f}")

report = os.path.join(OUTPUT_DIR, "anomaly_report.txt")
if os.path.exists(report):
    print("\n" + "=" * 60)
    print("  REPORT")
    print("=" * 60)
    with open(report, encoding="utf-8") as f:
        print(f.read())


  TRANSIT ANOMALY DETECTION — MULTI-AGENT (Notebook)
  Time:       16:05:34
  Model:      mistral-small-latest
  Algorithm:  IsolationForest
  Output:     /Users/matteo/Desktop/MultiAgents-Pipeline-ONLY-/output

  [supervisor] You are a data cleaning agent for tabular datasets used in downstream merge and anomaly detection ta
  [code_executor] Running 'data_loading_allarmi' (attempt 1)...

  ────────────────────────────────────────
  GENERATED CODE:
  ────────────────────────────────────────
    1 | import os
    2 | import json
    3 | import pandas as pd
    4 | import numpy as np
    5 | from pathlib import Path
    6 | 
    7 | def to_snake_case(name):
    8 |     s1 = ''.join(['_' + c.lower() if c.isupper() else c for c in name]).lstrip('_')
    9 |     s2 = ''.join(['_' + c if c.isalnum() and not prev.isalnum() else c for prev, c in zip(s1, s1[1:])]).lstrip('_')
   10 |     s3 = '_'.join([seg for seg in s2.split('_') if seg])
   11 |     return s3
   12 | 
   13 | def normalize_c

KeyboardInterrupt: 